# Grid-cell emergence — one-click Colab GPU run

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/everest-an/M1/blob/main/kaggle_kernels/grid_cell_emergence/grid_cell_colab.ipynb)

Trains a **GRU baseline** and the **MT-LNN recurrent core** as path integrators, then scores whether their hidden units develop **hexagonal grid cells** (Banino/Sorscher). The free Colab **T4** is enough — this job is small (~0.8M params).

**Before you run:** `Runtime → Change runtime type → Hardware accelerator: GPU (T4)`.

Then `Runtime → Run all`. A full 8000-step run takes roughly ~30–60 min on a T4; drop `GC_STEPS` to `3000` for a quick first pass.

## 1. Confirm a GPU is attached

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Set Runtime -> Change runtime type -> GPU (T4), then Run all again.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
!nvidia-smi -L

## 2. Fetch the repo (the MT-LNN core lives here)

In [ ]:
import os
if not os.path.exists('/content/M1'):
    !git clone --depth 1 https://github.com/everest-an/M1.git /content/M1
# matplotlib is the only extra dep on Colab (torch + numpy ship in the image).
!pip -q install matplotlib >/dev/null
print('repo ready ->', os.path.exists('/content/M1/kaggle_kernels/grid_cell_emergence/run_grid_cell.py'))

## 3. Hyperparameters (tuned for grid EMERGENCE)

`GC_PLACE_DOG=1` (difference-of-Gaussians target) is **the** emergence trigger — turn it to `0` to watch grids vanish. Edit any value below.

In [ ]:
import os
os.environ.update({
    'GC_PLACE_DOG':       '1',     # difference-of-Gaussians place target (emergence trigger)
    'GC_READOUT_DROPOUT': '0.5',   # Banino-style readout regulariser
    'GC_STEPS':           '8000',  # training iterations (3000 = quick pass)
    'GC_N_PLACE':         '512',   # place-cell targets
    'GC_EVAL_TRAJ':       '2000',  # trajectories for rate maps
    'GC_BATCH':           '256',
    'GC_SEQ_LEN':         '180',
    'GC_N_BINS':          '32',
    'GC_LR':              '1e-3',
    'GC_WD':              '1e-4',
    'GC_SEED':            '0',
    'WORK_DIR':           '/content/gridcell_out',
})
print('hyperparams set; WORK_DIR =', os.environ['WORK_DIR'])

## 4. Train + score (GRU baseline, then MT-LNN core)

In [ ]:
# run_grid_cell.py self-clones the repo into WORK_DIR/M1 for the MT-LNN import;
# PYTHONPATH points at our checkout as a fallback.
!cd /content/M1 && PYTHONPATH=/content/M1 python kaggle_kernels/grid_cell_emergence/run_grid_cell.py

## 5. Results — metrics + grid-cell figures

Read a positive result as **MTLNN** `grid_score_max > 0.3` with `n_units_score_gt_0.3 > 0`. The GRU row is the sanity baseline confirming the pipeline works.

In [ ]:
import json, glob, os
from IPython.display import Image, display

work = '/content/gridcell_out'
with open(os.path.join(work, 'grid_cell_metrics.json')) as f:
    metrics = json.load(f)

for tag, m in metrics['models'].items():
    if 'error' in m:
        print(f'{tag}: ERROR -> {m["error"]}')
        continue
    print(f"{tag:6s} grid_score_max={m['grid_score_max']:.4f}  mean={m['grid_score_mean']:.4f}  "
          f"#(>0.3)={m['n_units_score_gt_0.3']}  loss={m['final_loss']:.3f}")
print('elapsed_s =', metrics.get('elapsed_s'))

for png in sorted(glob.glob(os.path.join(work, '*.png'))):
    print('\n' + os.path.basename(png))
    display(Image(filename=png))

## 6. (optional) Download the results bundle

In [ ]:
import shutil
shutil.make_archive('/content/gridcell_results', 'zip', '/content/gridcell_out')
try:
    from google.colab import files
    files.download('/content/gridcell_results.zip')
except Exception as e:
    print('manual download: /content/gridcell_results.zip  (', e, ')')